# message的使用

## 1.消息格式

举例1：json格式

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint
import os

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model = "deepseek:deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL,
)

# 通过JSON初始化
messages1 = [
    {"role":"system", "content":"你是一个善于给出通俗易懂解释的AI助手"},
    {"role":"user", "content":"你好"},
    {"role":"assistant", "content":"你好，我能帮你做啥？"},
    {"role":"user", "content":"什么是机器学习？"}
]

res = model.invoke(messages1)
rprint(res.content)

举例2：消息对象列表

In [ ]:
# 通过JSON初始化
messages2 = [
    SystemMessage(content = "你是一个善于给出通俗易懂解释的AI助手"),
    HumanMessage(content = "你好"),
    AIMessage(content = "你好，我能帮你做啥？"),
    HumanMessage(content = "什么是深度学习?")
]

res = model.invoke(messages2)
rprint(res.content)

## 2.HumanMessage的使用

举例1:使用Deepseek官网平台

In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2、模型的初始化
model = init_chat_model(
    model = "deepseek-flash",
    model_provider = "deepseek",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
)

messages = [
    SystemMessage("你是一个信息抽取器。你会收到多条来自不同发言者的 user 消息。每条消息可能带有 name 字段。你的任务是：严格根据每条消息的 name 提取发言者及其观点，并输出 JSON。禁止使用“第一个人/第二个人”这种相对称呼。若某条消息没有 name，则输出 unknown。输出格式：{\"speakers\":[{\"name\":\"...\",\"claim\":\"...\"}]}"),
    HumanMessage(
        content="我认为 1+1=2",
        name="Bob"
    ),
    HumanMessage(
        content="我认为 1+1>2",
        name="Tom"
    ),
    HumanMessage(
        content="请列出谁说了什么，不要判断对错。",
        name="audience"
    )
]

response = model.invoke(messages)
print(response.content)  # {"speakers":[{"name":"unknown","claim":"我认为 1+1=2"},{"name":"unknown","claim":"我认为 1+1>2"}]}  --- name跟模型供应商有关系

## 3.AIMessage的使用

In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2、模型的初始化
model = init_chat_model(
    model = "deepseek-flash",
    model_provider = "deepseek",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
)

messages = [
    SystemMessage("你叫小智，是一名助人为乐的助手。"),
    HumanMessage("你好，好久不见，请介绍下你自己。")
]

response = model.invoke(messages)
rprint(response)


## 3.ToolMessage的使用

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    api_base=os.getenv("DEEPSEEK_BASE_URL"),
)

def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    return "阴雨连绵"

model_with_tools = model.bind_tools([get_weather])

# ① 让模型自己产生 tool_calls
messages = [HumanMessage(content="今天北京天气如何")]
ai_message = model_with_tools.invoke(messages)
rprint("模型返回的 tool_calls:", ai_message.tool_calls)
messages.append(ai_message)

# ② 用模型给的 id 回传工具结果
for tc in ai_message.tool_calls:
    result = get_weather(**tc["args"])
    messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))

# ③ 第二次调用，拿到自然语言回答
final = model_with_tools.invoke(messages)
rprint("最终回复:", final.content)